# 1주차. LLM Agent - Function Call, Tool Call 기본 구조

## 0. 목표

이번 주에는 실제 OpenAI 모델이 도구 호출을 선택하고, Python 도구 실행 결과를 받아 최종 답변을 만드는 흐름을 확인한다.


## 1. 준비

모든 코드는 실제 API를 호출한다. `.env`에 `OPENAI_API_KEY`가 없으면 바로 실패한다.


In [ ]:
import json
import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. 노트북을 repo 안에서 실행하세요.")

REPO_ROOT = find_repo_root(Path.cwd())
load_dotenv(REPO_ROOT / ".env", override=False)

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


## 2. 개념

Function call은 모델이 어떤 도구를 어떤 인자로 호출할지 구조화해 말하는 방식이다. Tool call은 그 구조화된 요청을 실제 Python 함수 실행과 연결하는 전체 흐름이다.


## 3. 기본 개념 실습

가장 작은 흐름은 "자연어 요청 → 일정 생성 tool call → 저장된 일정 확인"이다. 여기서는 생성 도구 하나만 등록해 모델이 어떤 arguments를 만드는지 본다.


In [ ]:
schedules: list[dict[str, Any]] = []

@tool("create_schedule", description="개인 일정을 생성한다. date는 YYYY-MM-DD, start_time은 HH:MM 형식이다.")
def create_schedule(title: str, date: str, start_time: str, attendees: list[str] | None = None) -> str:
    """Create a personal schedule."""
    schedule = {"id": f"schedule-{len(schedules) + 1}", "title": title, "date": date, "start_time": start_time, "attendees": attendees or []}
    schedules.append(schedule)
    return json.dumps({"ok": True, "schedule": schedule}, ensure_ascii=False)

nana_basic_agent = create_agent(
    model=make_model(),
    tools=[create_schedule],
    system_prompt="너는 개인 일정 메이트 나나다. 오늘은 2026-04-23이다. 상대 날짜는 이 날짜 기준으로 YYYY-MM-DD로 바꾼다. 일정 생성이 필요하면 반드시 create_schedule 도구를 호출한 뒤 짧게 답한다.",
)

student_request = "내일 10시에 민수와 회의 일정 잡아줘"
result = nana_basic_agent.invoke({"messages": [{"role": "user", "content": student_request}]})

print(final_text(result))
show_json(extract_tool_trace(result))
show_json(schedules)


## 4. 카나메이트 확장 예제

기본 일정 생성에 `list_schedules` 조회 기능을 붙인다. 이제 나나는 일정을 만들고, 같은 저장소에서 현재 일정 목록도 읽을 수 있다.


In [ ]:
@tool("list_schedules", description="현재 생성된 개인 일정 목록을 조회한다.")
def list_schedules() -> str:
    """List personal schedules."""
    return json.dumps({"ok": True, "schedules": schedules}, ensure_ascii=False)

nana_agent = create_agent(
    model=make_model(),
    tools=[create_schedule, list_schedules],
    system_prompt="너는 개인 일정 메이트 나나다. 오늘은 2026-04-23이다. 상대 날짜는 이 날짜 기준으로 YYYY-MM-DD로 바꾼다. 일정 생성이나 조회가 필요하면 반드시 알맞은 도구를 호출한 뒤 짧게 답한다.",
)

extension_request = "지금까지 만든 일정 목록 보여줘"
extension_result = nana_agent.invoke({"messages": [{"role": "user", "content": extension_request}]})

print(final_text(extension_result))
show_json(extract_tool_trace(extension_result))


## 5. 확장 예제 실행

같은 agent로 새 일정을 하나 더 만들고, 곧바로 목록 조회를 실행한다. 학생은 요청 문장을 바꿔보며 생성 trace와 조회 trace가 어떻게 달라지는지 확인한다.


In [ ]:
extension_create_request = "내일 14시에 지아와 체크인 일정 잡아줘"
extension_create_result = nana_agent.invoke({"messages": [{"role": "user", "content": extension_create_request}]})
extension_create_trace = extract_tool_trace(extension_create_result)

extension_list_request = "현재 일정 목록 보여줘"
extension_list_result = nana_agent.invoke({"messages": [{"role": "user", "content": extension_list_request}]})
extension_list_trace = extract_tool_trace(extension_list_result)

print(final_text(extension_create_result))
show_json(extension_create_trace)
print(final_text(extension_list_result))
show_json(extension_list_trace)

assert any(event["event"] == "tool_call" and event["tool_name"] == "create_schedule" for event in extension_create_trace)
assert any(event["event"] == "tool_call" and event["tool_name"] == "list_schedules" for event in extension_list_trace)
assert len(schedules) >= 1
print("1주차 확장 예제 실행 통과")


## 6. 코드 작성형 실습(`week01.py`)

이번 실습은 4-5번에서 실행한 "일정 생성 -> 목록 조회" 흐름을 같은 주차 과제 파일 `week01.py`의 `run_student_schedule_request`에서 구현하고 실행한다.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "week01.py").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. 노트북을 repo 안에서 실행하세요.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from week01 import extract_tool_trace, final_text, show_json, run_student_schedule_request


### 실행 예시

In [ ]:
student_request = "다음 주 월요일 오전 9시에 카나메이트 발표 연습 일정 잡아줘"
student_result = run_student_schedule_request(student_request)

print(student_result["answer"])
print(student_result["list_answer"])
show_json({
    "created_schedule": student_result["created_schedule"],
    "schedules": student_result["schedules"],
})
show_json({"create_trace": student_result["trace"], "list_trace": student_result["list_trace"]})

## 6-0. 실습 자동 점검

아래 셀은 모델 답변 문구가 아니라 trace, structured response, payload처럼 구조화된 값을 확인한다.

In [ ]:
assert student_result["created_schedule"] is not None
assert any(event["event"] == "tool_call" and event["tool_name"] == "create_schedule" for event in student_result["trace"])
assert any(event["event"] == "tool_call" and event["tool_name"] == "list_schedules" for event in student_result["list_trace"])
assert {"id", "title", "date", "start_time", "attendees"}.issubset(student_result["created_schedule"].keys())
assert any(schedule["id"] == student_result["created_schedule"]["id"] for schedule in student_result["schedules"])
print("1주차 코드 작성형 실습 통과")

## 6-1. 노트북 Gradio UI 실습

아래 셀은 `create_demo()`를 import하지 않고, 노트북 안에서 Gradio UI를 직접 구성한다. 버튼을 누르면 6번 실습에서 가져온 같은 주차의 `weekXX.py` 함수가 실행된다.

터미널에서 독립 실행형 UI를 확인하고 싶을 때는 아래 명령을 사용할 수 있다.

```bash
python week01.py
```

In [ ]:
import gradio as gr


def run_week01_notebook_ui(request: str):
    try:
        result = run_student_schedule_request(request)
        return (
            f"{result['answer']}\n\n{result['list_answer']}",
            {"created_schedule": result["created_schedule"], "schedules": result["schedules"]},
            {"create_trace": result["trace"], "list_trace": result["list_trace"]},
        )
    except Exception as exc:
        return str(exc), {}, {}


with gr.Blocks(title="KanaMate Week 1 Notebook") as demo:
    gr.Markdown("# Week 1 - Schedule Tool Flow")
    request = gr.Textbox(label="요청", value="내일 10시에 민수와 회의 일정 잡아줘")
    run_button = gr.Button("실행", variant="primary")
    answer = gr.Textbox(label="모델 최종 답변", lines=5)
    result_json = gr.JSON(label="완성 결과")
    trace_json = gr.JSON(label="Tool Trace")
    run_button.click(run_week01_notebook_ui, inputs=request, outputs=[answer, result_json, trace_json])

demo.launch()

## 7. 회고

이번 주에는 모델 답변보다 tool call trace를 먼저 읽는 습관을 만들었다. 다음 주에는 모델 출력 자체를 Pydantic 구조로 검증한다.
